# 01 — Exploratory Data Analysis
**ESCP Hackathon 2026 · Flood Risk Prediction · Group 8**

**Group 8 assignment:**
- 🟢 **Training region: Severn** ← primary EDA focus
- 🔵 **Test region: Northumbria** ← inspected only for domain shift

> Rule: do not use region-specific knowledge about Northumbria that cannot
> be interpolated from the features themselves.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings, json
from pathlib import Path

try:
    import netCDF4 as nc
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "netCDF4", "-q"])
    import netCDF4 as nc

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"

# ── Group 8 config ────────────────────────────────────────────────────────────
TRAIN_REGION = "severn"
TEST_REGION  = "northumbria"

DATA_DIR = Path("../Data")

FILES = {
    "train_terrain": DATA_DIR / f"flood_risk_terrain_{TRAIN_REGION}.nc",
    "test_terrain":  DATA_DIR / f"flood_risk_terrain_{TEST_REGION}.nc",
    "train_weather": DATA_DIR / f"era5_land_{TRAIN_REGION}.nc",
    "test_weather":  DATA_DIR / f"era5_land_{TEST_REGION}.nc",
}

print(f"Group 8 — Train: {TRAIN_REGION.upper()} | Test: {TEST_REGION.upper()}")
print()
for label, path in FILES.items():
    exists = path.exists()
    size   = f"{path.stat().st_size/1e6:.0f} MB" if exists else "NOT FOUND"
    status = "✅" if exists else "❌"
    print(f"  {status} [{label:15s}]  {path.name}  ({size})")


## 1. Helper utilities

In [ ]:
def detect_xy_coords(ds):
    """Auto-detect x/y coordinate variable names in a NetCDF file."""
    X_CANDIDATES = ["projection_x_coordinate", "x", "lon", "longitude", "easting"]
    Y_CANDIDATES = ["projection_y_coordinate", "y", "lat", "latitude",  "northing"]
    
    varnames = list(ds.variables.keys())
    print(f"  All variables in file: {varnames}")
    
    xname = next((c for c in X_CANDIDATES if c in varnames), None)
    yname = next((c for c in Y_CANDIDATES if c in varnames), None)
    
    # fallback: first two 1-D numeric variables that aren't time
    if xname is None or yname is None:
        candidates = [v for v in varnames
                      if ds.variables[v].ndim == 1
                      and str(ds.variables[v].dtype).startswith("f")
                      and v not in ("time","valid_time")]
        if len(candidates) >= 2:
            xname = xname or candidates[0]
            yname = yname or candidates[1]
    
    if xname is None or yname is None:
        raise ValueError(f"Cannot find x/y coords. Variables available: {varnames}")
    
    print(f"  Detected coords → x='{xname}'  y='{yname}'")
    return xname, yname


def terrain_to_df(path, sample_frac=0.05, seed=42):
    """Load terrain NetCDF as a flat sampled DataFrame."""
    ds      = nc.Dataset(path, "r")
    xname, yname = detect_xy_coords(ds)
    
    x = ds.variables[xname][:]
    y = ds.variables[yname][:]
    n = len(x)
    
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(n, size=int(n * sample_frac), replace=False))

    data = {"x": np.array(x)[idx], "y": np.array(y)[idx]}
    skip = {xname, yname}
    for v in ds.variables:
        if v in skip: continue
        arr = ds.variables[v][:]
        flat = arr.flatten() if hasattr(arr, "flatten") else np.array(arr).flatten()
        if len(flat) == n:
            data[v] = flat[idx]
        # skip variables whose length doesn't match pixel count (e.g. time)

    ds.close()

    df = pd.DataFrame(data)
    df.replace(-9999,       np.nan, inplace=True)
    df.replace(9.96921e+36, np.nan, inplace=True)
    print(f"  Loaded {len(df):,} rows ({sample_frac*100:.0f}% sample) — {Path(path).name}")
    return df


## 2. Terrain data — variable inventory

We inspect both regions but pay close attention to **Severn (train)**.


In [ ]:
print("🟢 TRAINING REGION")
nc_summary(FILES["train_terrain"], label="Severn — flood_risk_terrain (TRAIN)")


In [ ]:
print("🔵 TEST REGION (inspect for domain shift only)")
nc_summary(FILES["test_terrain"], label="Northumbria — flood_risk_terrain (TEST)")


## 3. Weather data (ERA5) — variable inventory

In [ ]:
nc_summary(FILES["train_weather"], label="Severn — ERA5 weather (TRAIN)")
nc_summary(FILES["test_weather"],  label="Northumbria — ERA5 weather (TEST)")


## 4. Load terrain as flat DataFrames (5% sample)

Severn is ~4.4 GB so we sample. 5% is enough for EDA; full data used during training.


In [ ]:
print("Loading training data (Severn)...")
df_train = terrain_to_df(FILES["train_terrain"], sample_frac=0.05)

print("\nLoading test data (Northumbria) — for domain shift only...")
df_test  = terrain_to_df(FILES["test_terrain"],  sample_frac=0.05)

print(f"\nTrain shape : {df_train.shape}")
print(f"Test shape  : {df_test.shape}")
print(f"\nColumns: {list(df_train.columns)}")
display(df_train.describe().round(3))


## 5. Target variable — flood risk distributions (Severn / TRAIN)

We analyze target distributions only on Severn since that's what we train on.


In [ ]:
fig, axes = plt.subplots(1, len(RISK_VARS), figsize=(18, 4))
fig.suptitle("🟢 Severn (TRAIN) — Flood Risk Category Distribution by Depth",
             fontsize=12, fontweight="bold", color=TRAIN_COLOR)

for ax, rv in zip(axes, RISK_VARS):
    if rv not in df_train.columns:
        ax.set_visible(False); continue
    counts = df_train[rv].dropna().value_counts().sort_index()
    bars   = ax.bar([RISK_LABELS.get(int(k), k) for k in counts.index],
                    counts.values, color=COLORS[:len(counts)], edgecolor="white")
    ax.set_title(rv, fontsize=9)
    ax.tick_params(axis="x", labelsize=7, rotation=30)
    ax.tick_params(axis="y", labelsize=7)
    total = counts.sum()
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                f"{val/total*100:.1f}%", ha="center", fontsize=6)

plt.tight_layout()
plt.savefig("../reports/figures/01_risk_distributions_severn.png", bbox_inches="tight")
plt.show()

# Print class proportions
print("\nClass proportions in Severn (training set):")
for rv in RISK_VARS:
    if rv not in df_train.columns: continue
    vc = df_train[rv].dropna().value_counts(normalize=True).sort_index()*100
    print(f"  {rv}: " + "  ".join([f"{RISK_LABELS.get(int(k),k)}={v:.1f}%" for k,v in vc.items()]))


## 6. Spatial maps — flood risk at 0.2m

In [ ]:
cmap = mcolors.ListedColormap(["#FFFFFF","#2196F3","#4CAF50","#FF9800","#F44336"])
norm = mcolors.BoundaryNorm([0,0.5,1.5,2.5,3.5,4.5], cmap.N)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Spatial Flood Risk at 0.2m (5% sample)", fontsize=12, fontweight="bold")

for ax, df, title, color in [
    (axes[0], df_train, f"🟢 Severn (TRAIN)",        TRAIN_COLOR),
    (axes[1], df_test,  f"🔵 Northumbria (TEST)",     TEST_COLOR),
]:
    if "risk_0_2m" not in df.columns:
        ax.set_title(f"{title} — missing"); continue
    valid = df.dropna(subset=["risk_0_2m"])
    sc = ax.scatter(valid["x"], valid["y"], c=valid["risk_0_2m"],
                    s=0.3, cmap=cmap, norm=norm, rasterized=True)
    ax.set_title(title, fontsize=11, color=color, fontweight="bold")
    ax.set_xlabel("Easting (m)", fontsize=8)
    ax.set_ylabel("Northing (m)", fontsize=8)
    ax.tick_params(labelsize=7)
    cbar = plt.colorbar(sc, ax=ax, ticks=[1,2,3,4])
    cbar.ax.set_yticklabels(["Very Low","Low","Medium","High"], fontsize=7)

plt.tight_layout()
plt.savefig("../reports/figures/02_spatial_risk_map.png", bbox_inches="tight")
plt.show()


## 7. Terrain feature distributions — Severn (TRAIN)

Deep dive on features we will use to train the model.


In [ ]:
TERRAIN_VARS = ["dtm", "waw", "imd", "clc_type", "flow_acc", "rciw"]

fig, axes = plt.subplots(1, len(TERRAIN_VARS), figsize=(20, 4))
fig.suptitle("🟢 Severn (TRAIN) — Terrain Feature Distributions",
             fontsize=12, fontweight="bold", color=TRAIN_COLOR)

for ax, var in zip(axes, TERRAIN_VARS):
    if var not in df_train.columns:
        ax.set_visible(False); continue
    data = df_train[var].dropna()
    if data.nunique() <= 20:
        vc = data.value_counts().sort_index()
        ax.bar(vc.index.astype(str), vc.values, color=TRAIN_COLOR, edgecolor="white", alpha=0.8)
        ax.tick_params(axis="x", labelsize=6, rotation=45)
    else:
        vals = np.log1p(data) if var == "flow_acc" else data
        ax.hist(vals, bins=50, color=TRAIN_COLOR, edgecolor="white", alpha=0.8)
        if var == "flow_acc": ax.set_xlabel("log(1+flow_acc)", fontsize=7)
    ax.set_title(var, fontsize=9)
    ax.tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.savefig("../reports/figures/03_terrain_features_severn.png", bbox_inches="tight")
plt.show()


## 8. Domain shift analysis — Severn vs Northumbria

Critical for Group 8: we train on Severn and predict on Northumbria.
Any feature with very different distributions across regions will hurt generalization.


In [ ]:
COMPARE_VARS = ["dtm", "imd", "flow_acc", "waw"]

fig, axes = plt.subplots(1, len(COMPARE_VARS), figsize=(18, 4))
fig.suptitle("Domain Shift: Severn (train) vs Northumbria (test)",
             fontsize=12, fontweight="bold")

for ax, var in zip(axes, COMPARE_VARS):
    for df, label, color in [
        (df_train, f"Severn (train)",       TRAIN_COLOR),
        (df_test,  f"Northumbria (test)",   TEST_COLOR),
    ]:
        if var not in df.columns: continue
        vals = df[var].dropna()
        if var == "flow_acc":
            vals = np.log1p(vals)
        if df[var].nunique() <= 10:
            vc = vals.value_counts(normalize=True).sort_index()
            ax.plot(vc.index, vc.values, "o-", label=label, color=color)
        else:
            ax.hist(vals, bins=60, alpha=0.5, label=label, color=color,
                    density=True, edgecolor="none")
    title = f"log(1+{var})" if var=="flow_acc" else var
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig("../reports/figures/04_domain_shift.png", bbox_inches="tight")
plt.show()

# Quantify shift with mean/std comparison
print("\nFeature shift summary (Severn → Northumbria):")
print(f"{'Variable':<15} {'Severn mean':>12} {'North mean':>12} {'Severn std':>12} {'North std':>12}")
print("-"*65)
for var in COMPARE_VARS:
    if var not in df_train.columns or var not in df_test.columns: continue
    sv = df_train[var].dropna()
    nv = df_test[var].dropna()
    if var == "flow_acc":
        sv, nv = np.log1p(sv), np.log1p(nv)
    print(f"{var:<15} {sv.mean():>12.3f} {nv.mean():>12.3f} {sv.std():>12.3f} {nv.std():>12.3f}")


## 9. Elevation — the key domain shift problem

Northumbria sits ~100m higher on average than Severn.
Using raw `dtm` as a feature will make the model learn "high elevation = low risk in Severn"
but that won't transfer to Northumbria which is uniformly higher.
**Use relative elevation or rely on `flow_acc` instead.**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: raw elevation overlay
ax = axes[0]
for df, label, color in [(df_train,"Severn (train)",TRAIN_COLOR),(df_test,"Northumbria (test)",TEST_COLOR)]:
    if "dtm" not in df.columns: continue
    vals = df["dtm"].dropna() / 10  # dm → m
    ax.hist(vals, bins=80, alpha=0.5, label=f"{label}\nmean={vals.mean():.0f}m", color=color, density=True)
ax.set_xlabel("Elevation (m)"); ax.set_ylabel("Density")
ax.set_title("Raw Elevation — HIGH SHIFT ⚠️", fontsize=10)
ax.legend(fontsize=8)

# Right: relative elevation (z-score within region)
ax = axes[1]
for df, label, color in [(df_train,"Severn (train)",TRAIN_COLOR),(df_test,"Northumbria (test)",TEST_COLOR)]:
    if "dtm" not in df.columns: continue
    vals = df["dtm"].dropna()
    z    = (vals - vals.mean()) / vals.std()
    ax.hist(z, bins=80, alpha=0.5, label=label, color=color, density=True)
ax.set_xlabel("Z-score elevation"); ax.set_ylabel("Density")
ax.set_title("Relative Elevation (Z-score) — ALIGNED ✅", fontsize=10)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../reports/figures/05_elevation_domain_shift.png", bbox_inches="tight")
plt.show()

print("ACTION: Add relative elevation feature in src/features.py:")
print("  df['dtm_relative'] = (df['dtm'] - df['dtm'].mean()) / df['dtm'].std()")


## 10. Feature–target correlations (Severn / TRAIN only)

Spearman correlation — robust to non-linear monotonic relationships and outliers.


In [ ]:
TARGET    = "risk_0_2m"
FEAT_COLS = ["dtm", "waw", "imd", "flow_acc", "clc_type"]

sub  = df_train[[TARGET] + FEAT_COLS].dropna()
corr = sub.corr(method="spearman")[TARGET].drop(TARGET).sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5))
bar_colors = [TRAIN_COLOR if v > 0 else TEST_COLOR for v in corr.values]
ax.barh(corr.index, corr.values, color=bar_colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Spearman ρ with risk_0_2m", fontsize=10)
ax.set_title("🟢 Feature Correlations — Severn (TRAIN)", fontsize=11)
for i, (idx, val) in enumerate(corr.items()):
    ax.text(val + 0.002 if val >= 0 else val - 0.002, i,
            f"{val:.3f}", va="center", ha="left" if val>=0 else "right", fontsize=8)
plt.tight_layout()
plt.savefig("../reports/figures/06_feature_correlation_severn.png", bbox_inches="tight")
plt.show()

# Across all risk depths
print("\nSpearman correlations across all flood depths (Severn):")
corr_all = df_train[[c for c in FEAT_COLS+RISK_VARS if c in df_train.columns]].corr(method="spearman")
display(corr_all.loc[FEAT_COLS, [r for r in RISK_VARS if r in corr_all.columns]].round(3))


## 11. Class imbalance — all depths (Severn / TRAIN)

In [ ]:
records = []
for rv in RISK_VARS:
    if rv not in df_train.columns: continue
    vc = df_train[rv].dropna().value_counts(normalize=True).sort_index()*100
    nan_pct = df_train[rv].isna().mean()*100
    for k, v in vc.items():
        records.append({"depth": rv, "category": RISK_LABELS.get(int(k),k), "pct": round(v,2)})
    records.append({"depth": rv, "category": "NaN (no risk)", "pct": round(nan_pct,2)})

pivot = (pd.DataFrame(records)
           .pivot(index="depth", columns="category", values="pct")
           .fillna(0))

fig, ax = plt.subplots(figsize=(10, 4))
pivot.plot(kind="bar", ax=ax,
           color=["#90CAF9","#2196F3","#4CAF50","#FF9800","#F44336"],
           edgecolor="white", width=0.7)
ax.set_title("🟢 Severn (TRAIN) — Class Imbalance Across All Flood Depths",
             fontsize=11, fontweight="bold")
ax.set_ylabel("% of pixels"); ax.set_xlabel("")
ax.tick_params(axis="x", rotation=30, labelsize=9)
ax.legend(fontsize=8, bbox_to_anchor=(1.01,1))
plt.tight_layout()
plt.savefig("../reports/figures/07_class_imbalance_severn.png", bbox_inches="tight")
plt.show()

print("\nTIP: Very Low dominates → use class_weight='balanced' or focal loss in your model.")


## 12. Missing values (Severn / TRAIN)

In [ ]:
nan_pct = df_train.isna().mean().sort_values(ascending=True)*100
nan_pct = nan_pct[nan_pct > 0]

fig, ax = plt.subplots(figsize=(8, max(3, len(nan_pct)*0.35)))
if len(nan_pct) == 0:
    ax.text(0.5, 0.5, "No missing values in non-risk columns", ha="center", fontsize=11)
else:
    ax.barh(nan_pct.index, nan_pct.values, color=TRAIN_COLOR, edgecolor="white", alpha=0.8)
    ax.set_xlabel("% NaN")
ax.set_title("🟢 Severn (TRAIN) — Missing Values by Variable", fontsize=11)
plt.tight_layout()
plt.savefig("../reports/figures/08_missing_values_severn.png", bbox_inches="tight")
plt.show()

print("\nNaN in risk vars = pixel has no flood risk (outside flood zone).")
print("Decision needed: treat as class 0 or exclude from training?")


## 13. Weather data — Severn time series overview

ERA5 data at ~9.45km resolution. We'll engineer time-invariant extreme features
from this for the model (rolling windows, percentiles).


In [ ]:
def weather_spatial_mean(path, var):
    ds = nc.Dataset(path, "r")
    if var not in ds.variables:
        ds.close(); return None
    try:
        t     = ds.variables["valid_time"][:]
        dates = pd.to_datetime([str(nc.num2date(d, ds.variables["valid_time"].units)) for d in t])
    except:
        dates = pd.RangeIndex(len(ds.variables["valid_time"][:]))
    data = ds.variables[var][:]
    ds.close()
    ts = np.ma.mean(data, axis=tuple(range(1, data.ndim)))
    return pd.Series(ts.filled(np.nan), index=dates, name=var)

WEATHER_PLOTS = [
    ("tp",         "Total Precipitation (m/day)"),
    ("sro",        "Surface Runoff (m/day)"),
    ("swvl1_mean", "Soil Water Layer 1 (mean)"),
    ("t2m_mean",   "2m Temperature K (mean)"),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 8))
fig.suptitle("🟢 Severn (TRAIN) — ERA5 Weather Monthly Means", fontsize=12, fontweight="bold")

for ax, (var, label) in zip(axes.flatten(), WEATHER_PLOTS):
    ts = weather_spatial_mean(FILES["train_weather"], var)
    if ts is None:
        ax.set_title(f"{label}\n(not found)"); continue
    monthly = ts.resample("ME").mean()
    ax.plot(monthly.index, monthly.values, color=TRAIN_COLOR, linewidth=1.5)
    ax.fill_between(monthly.index, monthly.values, alpha=0.15, color=TRAIN_COLOR)
    ax.set_title(label, fontsize=9)
    ax.tick_params(labelsize=7)
    ax.set_xlabel("")

plt.tight_layout()
plt.savefig("../reports/figures/09_weather_severn.png", bbox_inches="tight")
plt.show()


## 14. Export EDA summary JSON (share with Claude / teammates)

In [ ]:
def extract_summary(path, label, sample_n=200_000):
    ds  = nc.Dataset(path, "r")
    out = {"label": label, "file": Path(path).name,
           "dimensions": {k: int(v.size) for k,v in ds.dimensions.items()},
           "variables": {}}
    for vname, var in ds.variables.items():
        info = {"shape": list(var.shape), "dtype": str(var.dtype)}
        if vname in ("time","valid_time"):
            try:
                t = var[:]
                info["range"]   = [str(nc.num2date(t[0],var.units)), str(nc.num2date(t[-1],var.units))]
                info["n_steps"] = int(len(t))
            except: pass
        elif vname in ("projection_x_coordinate","projection_y_coordinate","x","y"):
            arr = var[:]
            info["range"] = [round(float(arr.min()),1), round(float(arr.max()),1)]
        else:
            try:
                raw  = var[:]
                flat = np.ma.compressed(np.ma.masked_invalid(raw.flatten()))
                idx  = np.random.choice(len(flat), min(sample_n,len(flat)), replace=False)
                s    = flat[idx]
                uniq = np.unique(flat)
                info["stats"] = {
                    "min":     round(float(s.min()),5),
                    "max":     round(float(s.max()),5),
                    "mean":    round(float(s.mean()),5),
                    "std":     round(float(s.std()),5),
                    "p25":     round(float(np.percentile(s,25)),5),
                    "p75":     round(float(np.percentile(s,75)),5),
                    "pct_nan": round(100*(1-len(flat)/raw.size),2),
                }
                if len(uniq) <= 25:
                    info["unique_values"] = [round(float(v),3) for v in uniq]
            except Exception as e:
                info["error"] = str(e)
        out["variables"][vname] = info
    ds.close()
    return out

summary = {
    "group":          8,
    "train_region":   TRAIN_REGION,
    "test_region":    TEST_REGION,
    "terrain_train":  extract_summary(FILES["train_terrain"],  "Severn terrain (TRAIN)"),
    "terrain_test":   extract_summary(FILES["test_terrain"],   "Northumbria terrain (TEST)"),
    "weather_train":  extract_summary(FILES["train_weather"],  "Severn ERA5 (TRAIN)"),
    "weather_test":   extract_summary(FILES["test_weather"],   "Northumbria ERA5 (TEST)"),
}

out_path = Path("../reports/eda_summary.json")
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"✅  Saved: {out_path}  ({out_path.stat().st_size/1024:.1f} KB)")
print("Paste eda_summary.json back to Claude for feature engineering & modelling advice.")


## 15. Key takeaways for Group 8

| # | Finding | Action in model |
|---|---------|----------------|
| 1 | **Train=Severn, Test=Northumbria** | Never leak Northumbria labels into training |
| 2 | **Northumbria ~100m higher elevation** | Use `dtm_relative` (z-score) not raw `dtm` |
| 3 | **`flow_acc` is the safest terrain proxy** | Always include; use `log1p` transform |
| 4 | **Heavy class imbalance** (Very Low dominates) | `class_weight='balanced'` or focal loss |
| 5 | **Spatial correlation** — pixels are neighbors | CNN patches OR spatial lag features |
| 6 | **NaN in risk vars = no flood risk** | Decide: treat as class 0 OR exclude |
| 7 | **Weather at 9.45km, terrain at 20m** | Engineer time-invariant extremes, then join |
| 8 | **5 flood depths available** | Multi-output model (one model, 5 heads) |

### Next steps
1. Run `notebooks/02_weather_feature_engineering.ipynb` → build rolling-window features
2. Run `models/train_baseline.py` → Random Forest on terrain features only
3. Evaluate on Northumbria → measure cross-region generalization
4. Add weather features → retrain → compare
